In [9]:
import pandas as pd
import numpy as np
import os

DATA_DIR = r"G:\quant_data\daily_bar\market_data"

df_price = pd.read_csv(os.path.join(DATA_DIR,'close_matrix.csv'),index_col='Date',parse_dates=True)

print(df_price.shape)


(104, 10)


In [17]:
#收益率每日
returns = df_price.pct_change()
print(returns.shape)
returns1 = df_price / df_price.shift(1) -1
print(returns1.shape)

#动量计算
momentum_10 = df_price / df_price.shift(10) -1

#波动率计算
bodonglv_20 = returns.rolling(window = 20).std()

#缺失值处理
momentum_10 = momentum_10.dropna(how = 'all')
bodonglv_20 = bodonglv_20.dropna(how = 'all')

#结果检验
print(momentum_10.shape)
print(bodonglv_20.shape)

#保存
momentum_10.to_csv(os.path.join(DATA_DIR, 'momentum_10.csv'))
bodonglv_20.to_csv(os.path.join(DATA_DIR,'bodonglv_20.csv'))





(104, 10)
(104, 10)
(94, 10)
(84, 10)


In [21]:
import pandas as pd
import os

DATA_DIR = r"G:\quant_data\daily_bar\market_data"
df_dongliang = pd.read_csv(os.path.join(DATA_DIR,'momentum_10.csv'),
                           index_col = 'Date',
                          parse_dates = True)
df_mv = pd.read_csv(os.path.join(DATA_DIR,'mv_matrix.csv'),
                    index_col = 'trade_date',
                    parse_dates = True)

# 1. 强行将两个矩阵的行索引统一转换为无时区的纯日期对象 (date)
df_dongliang.index = pd.to_datetime(df_dongliang.index).date
df_mv.index = pd.to_datetime(df_mv.index).date

# 2. 强行将两个矩阵的列名转换为去除首尾空格的纯数字字符串
df_dongliang.columns = (
    df_dongliang.columns.astype(str).str.split(".").str[0].str.strip()
)
df_mv.columns = df_mv.columns.astype(str).str.split(".").str[0].str.strip()

def get_cross_sectional_spearman(
    df_factor1: pd.DataFrame, df_factor2: pd.DataFrame) -> pd.Series:
    #计算俩因子矩阵在时间截面上的每日Spearman相关系数序列
    common_idx = df_factor1.index.intersection(df_factor2.index)
    common_cols = df_factor1.columns.intersection(df_factor2.columns)

    f1 = df_factor1.loc[common_idx, common_cols].sort_index()
    f2 = df_factor2.loc[common_idx, common_cols].sort_index()

    # 矩阵化横截面计算，只返回序列结果，由调用层自行执行统计推断或中性化决策
    return f1.corrwith(f2, axis=1, method="spearman").dropna()

daily_corr_series = get_cross_sectional_spearman(df_dongliang,df_mv)

mean = daily_corr_series.mean()

if abs(mean) >= 0.2:
    print(abs(mean))
else:
    print(abs(mean))

0.10612508059316567


In [50]:
import pandas as pd
import os

DATA_DIR = r"G:\quant_data\daily_bar\market_data"
df_bodonglv = pd.read_csv(os.path.join(DATA_DIR,'bodonglv_20.csv'),
                           index_col = 'Date',
                          parse_dates = True)
df_mv = pd.read_csv(os.path.join(DATA_DIR,'mv_matrix.csv'),
                    index_col = 'trade_date',
                    parse_dates = True)

# 1. 强行将两个矩阵的行索引统一转换为无时区的纯日期对象 (date)
df_bodonglv.index = pd.to_datetime(df_bodonglv.index).date
df_mv.index = pd.to_datetime(df_mv.index).date

# 2. 强行将两个矩阵的列名转换为去除首尾空格的纯数字字符串
df_bodonglv.columns = (
    df_bodonglv.columns.astype(str).str.split(".").str[0].str.strip()
)
df_mv.columns = df_mv.columns.astype(str).str.split(".").str[0].str.strip()

def get_cross_sectional_spearman(
    df_factor1: pd.DataFrame, df_factor2: pd.DataFrame) -> pd.Series:
    #计算俩因子矩阵在时间截面上的每日Spearman相关系数序列
    common_idx = df_factor1.index.intersection(df_factor2.index)
    common_cols = df_factor1.columns.intersection(df_factor2.columns)

    f1 = df_factor1.loc[common_idx, common_cols].sort_index()
    f2 = df_factor2.loc[common_idx, common_cols].sort_index()

    # 矩阵化横截面计算，只返回序列结果，由调用层自行执行统计推断或中性化决策
    return f1.corrwith(f2, axis=1, method="spearman").dropna()

daily_corr_series = get_cross_sectional_spearman(df_bodonglv,df_mv)

mean = daily_corr_series.mean()

if abs(mean) >= 0.2:
    print(abs(mean))
    print('中性化')
    df_volatility_pure = get_market_cap_neutral_factor(df_dongliang, df_mv)
else:
    print(abs(mean))

0.2421356421356421
中性化


In [47]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def get_market_cap_neutral_factor(
    df_bodonglv: pd.DataFrame, df_mv: pd.DataFrame
) -> pd.DataFrame:
    # 1. 严格矩阵对齐（并在函数内部实施防御性重对齐，确保万无一失）
    # 强行规范化两个传入矩阵的索引
    bodonglv_cleaned = df_bodonglv.copy()
    mv_cleaned = df_mv.copy()
    bodonglv_cleaned.index = pd.to_datetime(bodonglv_cleaned.index).normalize()
    mv_cleaned.index = pd.to_datetime(mv_cleaned.index).normalize()

    common_idx = bodonglv_cleaned.index.intersection(mv_cleaned.index)
    common_cols = bodonglv_cleaned.columns.intersection(mv_cleaned.columns)

    f = bodonglv_cleaned.loc[common_idx, common_cols].sort_index()
    mv = mv_cleaned.loc[common_idx, common_cols].sort_index()

    # 2. 核心：市值取自然对数
    ln_mv = np.log(mv)

    # 3. 初始化残差矩阵
    df_neutral = pd.DataFrame(index=f.index, columns=f.columns, dtype=float)

    # 4. 逐日横截面 OLS 回归
    for date in f.index:
        y_sec = f.loc[date]
        x_sec = ln_mv.loc[date]

        valid_mask = y_sec.notnull() & x_sec.notnull()
        if valid_mask.sum() < 3:
            continue

        Y = y_sec[valid_mask].values
        X = sm.add_constant(x_sec[valid_mask].values)

        model = sm.OLS(Y, X)
        results = model.fit()
        df_neutral.loc[date, valid_mask] = results.resid

    # 5. 横截面中位数填充与类型推断
    df_neutral = df_neutral.infer_objects(copy=False)
    df_neutral = df_neutral.apply(
        lambda row: row.fillna(row.median() if row.notnull().any() else 0.0),
        axis=1,
    ).astype(float)

    # 6. 横截面 Z-score 标准化
    sec_mean = df_neutral.mean(axis=1)
    sec_std = df_neutral.std(axis=1)
    df_neutral = df_neutral.sub(sec_mean, axis=0).div(sec_std + 1e-8, axis=0)

    # 🌟 核心修正：只纯粹返回计算完成的矩阵对象，把打印和保存移到外面
    return df_neutral
    
# 1. 外部调用函数，用变量承接返回的 DataFrame 矩阵
df_neutral_result = get_market_cap_neutral_factor(df_bodonglv, df_mv)

df_neutral_result.index.name = "Date"

# 2. 在外部执行你需要的保存或验证逻辑
df_neutral_result.to_csv(os.path.join(DATA_DIR, "neutral.csv"))

In [49]:
import pandas as pd
import os

DATA_DIR = r"G:\quant_data\daily_bar\market_data"
df_neutral = pd.read_csv(os.path.join(DATA_DIR,'neutral.csv'),
                           index_col = 'Date',
                          parse_dates = True)
df_mv = pd.read_csv(os.path.join(DATA_DIR,'mv_matrix.csv'),
                    index_col = 'trade_date',
                    parse_dates = True)

# 1. 强行将两个矩阵的行索引统一转换为无时区的纯日期对象 (date)
df_neutral.index = pd.to_datetime(df_neutral.index).date
df_mv.index = pd.to_datetime(df_mv.index).date

# 2. 强行将两个矩阵的列名转换为去除首尾空格的纯数字字符串
df_neutral.columns = (
    df_neutral.columns.astype(str).str.split(".").str[0].str.strip()
)
df_mv.columns = df_mv.columns.astype(str).str.split(".").str[0].str.strip()

def get_cross_sectional_spearman(
    df_factor1: pd.DataFrame, df_factor2: pd.DataFrame) -> pd.Series:
    #计算俩因子矩阵在时间截面上的每日Spearman相关系数序列
    common_idx = df_factor1.index.intersection(df_factor2.index)
    common_cols = df_factor1.columns.intersection(df_factor2.columns)

    f1 = df_factor1.loc[common_idx, common_cols].sort_index()
    f2 = df_factor2.loc[common_idx, common_cols].sort_index()

    # 矩阵化横截面计算，只返回序列结果，由调用层自行执行统计推断或中性化决策
    return f1.corrwith(f2, axis=1, method="spearman").dropna()

daily_corr_series = get_cross_sectional_spearman(df_neutral,df_mv)

mean = daily_corr_series.mean()

if abs(mean) >= 0.2:
    print(abs(mean))
else:
    print(abs(mean))
    print('合格')

0.1080808080808081
合格
